# Verification των duplicate tensors του ZooLake

Το notebook ελέγχει το περιεχόμενο των image tensors του Data.pickle,
εντοπίζει επαναλαμβανόμενες εγγραφές και επιβεβαιώνει τον καθαρισμένο
διαχωρισμό train, validation και test.

In [1]:
from pathlib import Path, PurePosixPath
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

## 1. Εντοπισμός αρχείων

In [2]:
KAGGLE_INPUT = Path("/kaggle/input")

data_pickle_candidates = sorted(
    KAGGLE_INPUT.rglob("Data.pickle")
)

classes_candidates = sorted(
    KAGGLE_INPUT.rglob("classes_ERIC.npy")
)

print("Data.pickle candidates:")
for path in data_pickle_candidates:
    print(" -", path)

print("\nclasses_ERIC.npy candidates:")
for path in classes_candidates:
    print(" -", path)

DATA_PICKLE_PATH = data_pickle_candidates[0]
CLASSES_PATH = classes_candidates[0]

print("\nSelected Data.pickle:", DATA_PICKLE_PATH)
print("Selected classes file:", CLASSES_PATH)

Data.pickle candidates:
 - /kaggle/input/datasets/tsikaripunks/data-pickle-paper-split/Data.pickle

classes_ERIC.npy candidates:
 - /kaggle/input/datasets/tsikaripunks/classes/classes_ERIC.npy

Selected Data.pickle: /kaggle/input/datasets/tsikaripunks/data-pickle-paper-split/Data.pickle
Selected classes file: /kaggle/input/datasets/tsikaripunks/classes/classes_ERIC.npy


## 2. Φόρτωση δεδομένων

In [11]:
Data = pd.read_pickle(DATA_PICKLE_PATH)


def decode_class_name(value):
    if isinstance(value, bytes):
        value = value.decode("utf-8")

    value = str(value).strip().lower()

    aliases = {
        "kellikottia": "kellicottia",
    }

    return aliases.get(value, value)


CLASS_NAMES = [
    decode_class_name(value)
    for value in np.load(
        CLASSES_PATH,
        allow_pickle=True,
    ).tolist()
]

assert len(CLASS_NAMES) == 35

print("\nNumber of classes:", len(CLASS_NAMES))
print("Number of Data.pickle elements:", len(Data))

# Data[0:3] → train
# Data[3:6] → validation
# Data[6:9] → final test
# με βάση τον κώδικα των ερευνητών.


split_sources = [
    (
        "train",
        Data[0],
        Data[1],
        Data[2],
    ),
    (
        "validation",
        Data[3],
        Data[4],
        Data[5],
    ),
    (
        "test",
        Data[6],
        Data[7],
        Data[8],
    ),
]




Number of classes: 35
Number of Data.pickle elements: 12


## 3. SHA-256 κάθε tensor

In [4]:
def tensor_sha256(image):
    array = np.ascontiguousarray(
        np.asarray(image)
    )

    hasher = hashlib.sha256()

    # Ελέγχουμε dtype και shape όχι μόνο τα pixels.
    hasher.update(str(array.dtype).encode("utf-8"))
    hasher.update(
        np.asarray(
            array.shape,
            dtype=np.int64,
        ).tobytes()
    )
    hasher.update(array.tobytes())

    return hasher.hexdigest()


rows = []
images_by_global_row = {}

global_row = 0

for (
    split_name,
    filenames,
    images,
    labels,
) in split_sources:

    assert len(filenames) == len(images) == len(labels)

    for row_in_split, (
        filename,
        image,
        encoded_label,
    ) in enumerate(
        zip(
            filenames,
            images,
            labels,
        )
    ):

        label_array = np.asarray(encoded_label)

        if label_array.size > 1:
            class_index = int(
                np.argmax(label_array)
            )
        else:
            class_index = int(
                label_array.item()
            )

        source_path = str(filename)
        normalized_path = source_path.replace("\\", "/")

        rows.append({
            "global_row": global_row,
            "row_in_split": row_in_split,
            "split": split_name,
            "class_index": class_index,
            "label": CLASS_NAMES[class_index],
            "filename": PurePosixPath(
                normalized_path
            ).name,
            "source_path": normalized_path,
            "tensor_shape": str(
                np.asarray(image).shape
            ),
            "tensor_dtype": str(
                np.asarray(image).dtype
            ),
            "tensor_sha256": tensor_sha256(
                image
            ),
        })

        images_by_global_row[global_row] = image

        global_row += 1


manifest = pd.DataFrame(rows)

print("\nOriginal split counts:")
print(
    manifest["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nTotal entries:", len(manifest))
print(
    "Unique tensor hashes:",
    manifest["tensor_sha256"].nunique(),
)




Original split counts:
split
train         12560
validation     2692
test           2691
Name: count, dtype: int64

Total entries: 17943
Unique tensor hashes: 17940


## 4. Εντοπισμός των duplicate groups

In [5]:
hash_counts = (
    manifest
    .groupby("tensor_sha256")
    .size()
    .sort_values(ascending=False)
)

duplicate_hash_counts = hash_counts[
    hash_counts > 1
]

duplicate_hashes = set(
    duplicate_hash_counts.index
)

repeated = manifest[
    manifest["tensor_sha256"].isin(
        duplicate_hashes
    )
].copy()

hash_to_group = {
    image_hash: f"D{index:02d}"
    for index, image_hash in enumerate(
        duplicate_hash_counts.index,
        start=1,
    )
}

repeated["duplicate_group"] = (
    repeated["tensor_sha256"]
    .map(hash_to_group)
)

number_of_duplicate_groups = len(
    duplicate_hash_counts
)

extra_duplicate_occurrences = int(
    (duplicate_hash_counts - 1).sum()
)

print(
    "\nDuplicate content groups:",
    number_of_duplicate_groups,
)

print(
    "Extra duplicate occurrences:",
    extra_duplicate_occurrences,
)

print(
    "Rows belonging to repeated content:",
    len(repeated),
)




Duplicate content groups: 3
Extra duplicate occurrences: 3
Rows belonging to repeated content: 6


## 5. Έλεγχος labels και ακριβούς ισότητας tensors

In [6]:
label_conflicts = (
    repeated
    .groupby("tensor_sha256")["class_index"]
    .nunique()
)

label_conflicts = label_conflicts[
    label_conflicts > 1
]

assert label_conflicts.empty, (
    "Βρέθηκε ίδιο image content με διαφορετικά labels."
)

for image_hash, group in repeated.groupby(
    "tensor_sha256"
):

    group_rows = group["global_row"].tolist()

    first_array = np.asarray(
        images_by_global_row[group_rows[0]]
    )

    for row_id in group_rows[1:]:
        other_array = np.asarray(
            images_by_global_row[row_id]
        )

        assert np.array_equal(
            first_array,
            other_array,
        ), (
            "Ίδιο SHA-256 αλλά διαφορετικά tensors. "
            "Αυτό δεν θα έπρεπε να είναι δυνατό."
        )

print("Label conflicts: 0")
print("Tensor equality check: PASS")


Label conflicts: 0
Tensor equality check: PASS


## 6. Ανακατασκευή leakage-free split

In [7]:
priority = {
    "test": 0,
    "validation": 1,
    "train": 2,
}

clean_manifest = (
    manifest
    .assign(
        _priority=manifest["split"].map(
            priority
        )
    )
    .sort_values(
        [
            "tensor_sha256",
            "_priority",
            "global_row",
        ],
        kind="stable",
    )
    .drop_duplicates(
        "tensor_sha256",
        keep="first",
    )
    .drop(columns="_priority")
    .sort_values("global_row")
    .reset_index(drop=True)
)

kept_global_rows = set(
    clean_manifest["global_row"]
)

removed = manifest[
    ~manifest["global_row"].isin(
        kept_global_rows
    )
].copy()

repeated["status"] = np.where(
    repeated["global_row"].isin(
        kept_global_rows
    ),
    "KEPT",
    "REMOVED",
)

repeated = repeated.sort_values(
    [
        "duplicate_group",
        "status",
        "split",
    ]
)

print("\nDuplicate groups and decision:")

display(
    repeated[[
        "duplicate_group",
        "status",
        "split",
        "label",
        "filename",
        "tensor_shape",
        "tensor_dtype",
        "tensor_sha256",
    ]]
)


print("\nRemoved occurrences:")

display(
    removed[[
        "split",
        "label",
        "filename",
        "tensor_sha256",
    ]]
)



Duplicate groups and decision:


,duplicate_group,status,split,label,filename,tensor_shape,tensor_dtype,tensor_sha256
13034,D01,KEPT,validation,dinobryon,SPC-EAWAG-0P5X-1559498570246055-6403994484253-...,"(128, 128, 3)",float32,ad31c248ea4b58ed53462a541aa5cc5d02c849cc97b603...
5727,D01,REMOVED,train,dinobryon,SPC-EAWAG-0P5X-1559498570246055-6403994484253-...,"(128, 128, 3)",float32,ad31c248ea4b58ed53462a541aa5cc5d02c849cc97b603...
3458,D02,KEPT,train,dinobryon,SPC-EAWAG-0P5X-1528850008503168-564006655548-0...,"(128, 128, 3)",float32,39d44242d88af8c510acb8b9604701223059edc9ebb00f...
7467,D02,REMOVED,train,dinobryon,SPC-EAWAG-0P5X-1528850008503168-564006655548-0...,"(128, 128, 3)",float32,39d44242d88af8c510acb8b9604701223059edc9ebb00f...
17397,D03,KEPT,test,rotifers,SPC-EAWAG-0P5X-1589537326874154-10282734293288...,"(128, 128, 3)",float32,16313bfd4fd753af1809caab4538055a4f105f2248bd92...
14310,D03,REMOVED,validation,rotifers,SPC-EAWAG-0P5X-1589537326874154-10282734293288...,"(128, 128, 3)",float32,16313bfd4fd753af1809caab4538055a4f105f2248bd92...



Removed occurrences:


,split,label,filename,tensor_sha256
5727,train,dinobryon,SPC-EAWAG-0P5X-1559498570246055-6403994484253-...,ad31c248ea4b58ed53462a541aa5cc5d02c849cc97b603...
7467,train,dinobryon,SPC-EAWAG-0P5X-1528850008503168-564006655548-0...,39d44242d88af8c510acb8b9604701223059edc9ebb00f...
14310,validation,rotifers,SPC-EAWAG-0P5X-1589537326874154-10282734293288...,16313bfd4fd753af1809caab4538055a4f105f2248bd92...


## 7. Έλεγχος διατήρησης μοναδικού περιεχομένου

In [8]:
original_unique_hashes = set(
    manifest["tensor_sha256"]
)

clean_unique_hashes = set(
    clean_manifest["tensor_sha256"]
)

unique_content_preserved = (
    original_unique_hashes
    == clean_unique_hashes
)

clean_counts = (
    clean_manifest["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nClean split counts:")
print(clean_counts)

print(
    "\nUnique image content preserved:",
    unique_content_preserved,
)

print(
    "Unique hashes before:",
    len(original_unique_hashes),
)

print(
    "Unique hashes after:",
    len(clean_unique_hashes),
)




Clean split counts:
split
train         12558
validation     2691
test           2691
Name: count, dtype: int64

Unique image content preserved: True
Unique hashes before: 17940
Unique hashes after: 17940


## 8. Τελικές assertions

In [9]:
assert len(manifest) == 17_943
assert len(original_unique_hashes) == 17_940
assert number_of_duplicate_groups == 3
assert extra_duplicate_occurrences == 3
assert len(removed) == 3
assert unique_content_preserved

expected_clean_counts = {
    "train": 12_558,
    "validation": 2_691,
    "test": 2_691,
}

assert clean_counts.to_dict() == expected_clean_counts, (
    f"Unexpected clean counts: {clean_counts.to_dict()}"
)

print("\nFINAL RESULT: PASS")
print(
    "Οι 17.943 entries αντιστοιχούν σε "
    "17.940 μοναδικά image tensors."
)
print(
    "Αφαιρέθηκαν μόνο 3 επαναλαμβανόμενες "
    "εμφανίσεις και δεν χάθηκε κανένα unique tensor."
)



FINAL RESULT: PASS
Οι 17.943 entries αντιστοιχούν σε 17.940 μοναδικά image tensors.
Αφαιρέθηκαν μόνο 3 επαναλαμβανόμενες εμφανίσεις και δεν χάθηκε κανένα unique tensor.


# 9. Αποθήκευση  CSV

In [10]:
OUTPUT_PATH = Path(
    "/kaggle/working/"
    "zoolake_duplicate_groups_verified.csv"
)

repeated.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("\nSaved evidence:", OUTPUT_PATH)


Saved evidence: /kaggle/working/zoolake_duplicate_groups_verified.csv
